# 01 — Food CPI sub-indices: hierarchy, coverage and usability per tariff episode

Next step 2 in `CONTRIBUTING.md`. Questions:
1. Which food CPI sub-indices exist in table 18-10-0004-01, and over which dates? (D03)
2. Which are available by province, Ontario included? (D13)
3. Which have enough history before each tariff episode for a synthetic control, and data after it? (D05, D17)
4. Are the sub-indices fine enough to isolate tariffed products? (D05)

Data: `data/raw/` from `src/fetch_data.py`. Logic: `src/cpi_catalog.py`. Output: `data/processed/cpi_food_catalog.csv` (not versioned).
French labels come from the FR table, joined on StatCan member ids (D16).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src") if Path.cwd().name == "notebooks" else "src")
import pandas as pd
import cpi_catalog as C
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 70)

cat = C.build_catalog()
out = C.ROOT / "data" / "processed" / "cpi_food_catalog.csv"
out.parent.mkdir(parents=True, exist_ok=True)
cat.to_csv(out, index=False)
print(f"{len(cat)} food products (nodes of the 'Food' subtree) -> {out.relative_to(C.ROOT)}")
print("Last published month:", cat.end.max())

129 food products (nodes of the 'Food' subtree) -> data\processed\cpi_food_catalog.csv
Last published month: 2026-08


## 1. Hierarchy and national coverage

In [2]:
view = cat.assign(product=cat.depth.map(lambda d: "  " * d) + cat["product"])
view[["product", "base", "start", "end", "n_months", "n_geo", "is_leaf"]]

,product,base,start,end,n_months,n_geo,is_leaf
0,Food,2002=100,1949-01,2026-08,932,13,False
1,Food purchased from stores,2002=100,1961-01,2026-08,788,13,False
2,Meat,2002=100,1978-09,2026-08,576,13,False
3,Fresh or frozen meat (excluding poultry),2002=100,1978-09,2026-08,576,13,False
4,Fresh or frozen beef,2002=100,1971-01,2026-08,668,13,False
5,Fresh or frozen beef hip cuts (202404=100),202404=100,2024-04,2026-08,29,1,True
6,Fresh or frozen beef loin cuts (202404=100),202404=100,2024-04,2026-08,29,1,True
7,Fresh or frozen beef rib cuts (202404=100),202404=100,2024-04,2026-08,29,1,True
8,Fresh or frozen beef chuck cuts (202404=100),202404=100,2024-04,2026-08,29,1,True
9,Fresh stewing beef (202404=100),202404=100,2024-04,2026-08,29,1,True


## 2. Coverage by index base

StatCan adds detailed series over time with their own base period (for example `2013=100`, `202304=100`). The base tells when a series starts.

In [3]:
by_base = (cat.groupby("base")
           .agg(n=("product", "size"), first_start=("start", "min"), last_start=("start", "max"),
                ended_before_latest=("end", lambda e: (e < cat.end.max()).sum()), max_geo=("n_geo", "max"))
           .sort_values("first_start"))
by_base

,n,first_start,last_start,ended_before_latest,max_geo
base,,,,,
2002=100,76,1949-01,1994-12,0,13
2013=100,30,2013-01,2013-01,5,1
202304=100,8,2023-04,2023-04,0,1
202404=100,15,2024-04,2024-04,0,1


In [4]:
print("Series discontinued before the latest month:")
cat.loc[cat.end < cat.end.max(), ["product", "base", "start", "end"]]

Series discontinued before the latest month:


,product,base,start,end
57,Crackers and crisp breads (2013=100),2013=100,2013-01,2021-05
58,Cookies and sweet biscuits (2013=100),2013=100,2013-01,2021-05
111,Fermented or pickled vegetables (2013=100),2013=100,2013-01,2021-05
112,"Sauces, condiments and dips (2013=100)",2013=100,2013-01,2021-05
113,"Herbs, spices and seasonings (2013=100)",2013=100,2013-01,2021-05


## 3. Provincial availability (D13)

In [5]:
prov = cat[cat.n_geo > 1]
print(f"{len(prov)} of {len(cat)} food series are published below the national level; all include Ontario: {prov.has_ontario.all()}")
print("Their base periods:", prov.base.unique().tolist())
prov[["product", "start", "n_geo"]]

32 of 129 food series are published below the national level; all include Ontario: True
Their base periods: ['2002=100']


,product,start,n_geo
0,Food,1949-01,13
1,Food purchased from stores,1961-01,13
2,Meat,1978-09,13
3,Fresh or frozen meat (excluding poultry),1978-09,13
4,Fresh or frozen beef,1971-01,13
12,Fresh or frozen pork,1978-09,13
17,Fresh or frozen poultry,1949-01,13
18,Fresh or frozen chicken,1949-01,13
25,Processed meat,1985-01,13
30,"Fish, seafood and other marine products",1985-01,13


In [6]:
# Which geographies publish the 'Food' index?
cpi = C.read_statcan("cpi_monthly", columns=["REF_DATE", "GEO", "Products and product groups", "UOM", "VALUE"])
food = cpi[(cpi["product"] == "Food") & (cpi.UOM == "2002=100")]
food.groupby("GEO").REF_DATE.agg(["min", "max"])

,min,max
GEO,,
Alberta,1978-09,2026-08
British Columbia,1978-09,2026-08
Canada,1949-01,2026-08
Manitoba,1978-09,2026-08
New Brunswick,1978-09,2026-08
Newfoundland and Labrador,1978-09,2026-08
Nova Scotia,1978-09,2026-08
Ontario,1978-09,2026-08
Prince Edward Island,1978-09,2026-08


## 4. Usability per tariff episode (D17)

A series is usable for an episode once it has at least `MIN_PRE_MONTHS` = 36 months before the first tariff month (`_pre_ok`) and at least one month after it (`_post` > 0).
Episodes come from `data/reference/tariff_events.csv`.

In [7]:
events = pd.read_csv(C.TARIFF_EVENTS)
rows = []
for ev in events.itertuples():
    pre_ok = cat[f"{ev.event_id}_pre_ok"]
    post = cat[f"{ev.event_id}_post"] > 0
    rows.append({"episode": ev.event_id, "start": ev.start_date, "pre_ok": pre_ok.sum(),
                 "post_available": post.sum(), "usable_now": (pre_ok & post).sum(),
                 "leaves_usable_now": (pre_ok & post & cat.is_leaf).sum()})
pd.DataFrame(rows)

,episode,start,pre_ok,post_available,usable_now,leaves_usable_now
0,CA2018,2018-07-01,106,129,106,66
1,CA2025,2025-03-04,106,124,101,61
2,CA2026,2026-09-08,114,0,0,0


In [8]:
print("Series NOT usable for 2025 because they start too late (too little history):")
cat.loc[~cat.CA2025_pre_ok, ["product", "base", "start", "CA2025_pre"]]

Series NOT usable for 2025 because they start too late (too little history):


,product,base,start,CA2025_pre
5,Fresh or frozen beef hip cuts (202404=100),202404=100,2024-04,11
6,Fresh or frozen beef loin cuts (202404=100),202404=100,2024-04,11
7,Fresh or frozen beef rib cuts (202404=100),202404=100,2024-04,11
8,Fresh or frozen beef chuck cuts (202404=100),202404=100,2024-04,11
9,Fresh stewing beef (202404=100),202404=100,2024-04,11
10,Fresh or frozen ground beef (202404=100),202404=100,2024-04,11
11,Other fresh or frozen beef (202404=100),202404=100,2024-04,11
13,Fresh or frozen pork loin cuts (202404=100),202404=100,2024-04,11
14,Fresh or frozen pork shoulder cuts (202404=100),202404=100,2024-04,11
15,Other fresh or frozen pork (202404=100),202404=100,2024-04,11


## 5. Granularity for the 2026 dairy and cheese tariffs (D05)

The September 2026 tariffs target dairy (50%) and cheeses (25%) [R7, R8]. Which series cover them, and at which level?

In [9]:
dairy = cat[cat.parent.isin(["Dairy products", "Fresh milk", "Cheese"]) | cat["product"].isin(["Dairy products"])]
dairy[["product", "product_fr", "base", "start", "n_geo", "CA2026_pre", "CA2026_pre_ok", "CA2026_post"]]

,product,product_fr,base,start,n_geo,CA2026_pre,CA2026_pre_ok,CA2026_post
40,Dairy products,Produits laitiers,2002=100,1949-01,13,932,True,0
41,Fresh milk,Lait frais,2002=100,1949-01,13,932,True,0
42,Whole milk (2013=100),Lait entier (2013=100),2013=100,2013-01,1,164,True,0
43,Low-fat milk (2013=100),Lait faible en matières grasses (2013=100),2013=100,2013-01,1,164,True,0
44,Butter,Beurre,2002=100,1949-01,13,932,True,0
45,Cheese,Fromage,2002=100,1985-01,13,500,True,0
46,Cheddar cheese (2013=100),Fromage cheddar (2013=100),2013=100,2013-01,1,164,True,0
47,Processed cheese (2013=100),Fromage fondu (2013=100),2013=100,2013-01,1,164,True,0
48,Unripened or fresh cheese (2013=100),Fromage non affiné ou fromage frais (2013=100),2013=100,2013-01,1,164,True,0
49,Mozzarella cheese (2013=100),Fromage mozzarella (2013=100),2013=100,2013-01,1,164,True,0


## 6. Findings

Written up in the decision log (D13 update, D17). See the text below each table for the numbers.